In [3]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os

fake = Faker('en_IN')
np.random.seed(42)
random.seed(42)
N = 5000

# --- create output folder if it doesn't exist ---
output_path = r'C:\Users\chant\ecommerce-analytics\data\raw'
os.makedirs(output_path, exist_ok=True)

# --- customers ---
pincodes = [500001,500002,500003,600001,600002,110001,110002,560001,700001,226001]
pincode_rural = {500001:False,500002:False,500003:False,600001:False,600002:False,
                 110001:False,110002:False,560001:True,700001:True,226001:True}
customers = pd.DataFrame({
    'customer_id': [f'C{i:05d}' for i in range(N)],
    'name': [fake.name() for _ in range(N)],
    'city': [fake.city() for _ in range(N)],
    'pincode': np.random.choice(pincodes, N),
    'household_size': np.random.randint(1, 8, N),
    'profession': np.random.choice(['salaried','business','student','homemaker'], N,
                                    p=[0.45,0.25,0.15,0.15])
})
customers['is_rural'] = customers['pincode'].map(pincode_rural)
customers.to_csv(os.path.join(output_path, 'customers.csv'), index=False)
print(f"customers: {len(customers):,} rows saved")

# --- products ---
categories = ['Electronics','Fashion','Beauty','Grocery','Home']
products = pd.DataFrame({
    'product_id': [f'P{i:04d}' for i in range(500)],
    'name': [fake.bs() for _ in range(500)],
    'category': np.random.choice(categories, 500, p=[0.2,0.25,0.15,0.25,0.15]),
    'price': np.round(np.random.exponential(800, 500) + 99, 2),
    'is_perishable': np.random.choice([True,False], 500, p=[0.25,0.75]),
    'box_size_used': np.random.choice(['S','M','L','XL'], 500, p=[0.1,0.3,0.4,0.2]),
    'actual_size_needed': np.random.choice(['S','M','L','XL'], 500, p=[0.35,0.35,0.2,0.1]),
    'stock_level': np.random.randint(0, 500, 500),
    'reorder_point': np.random.randint(20, 100, 500),
    'seller_id': np.random.choice([f'S{i:03d}' for i in range(50)], 500),
    'is_authentic': np.random.choice([True,False], 500, p=[0.88,0.12])
})
products.to_csv(os.path.join(output_path, 'products.csv'), index=False)
print(f"products:  {len(products):,} rows saved")

# --- orders ---
base_date = datetime(2024, 1, 1)
orders = pd.DataFrame({
    'order_id': [f'ORD{i:06d}' for i in range(N*2)],
    'customer_id': np.random.choice(customers['customer_id'], N*2),
    'product_id': np.random.choice(products['product_id'], N*2),
    'order_date': [base_date + timedelta(days=random.randint(0,365)) for _ in range(N*2)],
    'quantity': np.random.randint(1, 10, N*2),
    'base_price': np.round(np.random.exponential(700, N*2) + 99, 2),
    'hidden_fee': np.round(np.random.choice([0,0,0,49,99,149], N*2), 2),
    'discount_applied': np.round(np.random.choice([0,0,50,100,200], N*2), 2),
    'delivery_type': np.random.choice(['standard','quick','express'], N*2, p=[0.5,0.35,0.15]),
    'cart_abandoned': np.random.choice([True,False], N*2, p=[0.35,0.65]),
    'order_hour': [random.randint(0,23) for _ in range(N*2)]
})
orders['final_price'] = orders['base_price'] + orders['hidden_fee'] - orders['discount_applied']
orders.to_csv(os.path.join(output_path, 'orders.csv'), index=False)
print(f"orders:    {len(orders):,} rows saved")

# --- returns ---
return_reasons = ['wrong size','color mismatch','quality issue','not as shown',
                  'damaged','counterfeit suspected','changed mind']
num_returns = int(N * 0.18)
returns = pd.DataFrame({
    'return_id': [f'RET{i:05d}' for i in range(num_returns)],
    'order_id': np.random.choice(orders['order_id'], num_returns, replace=False),
    'reason': np.random.choice(return_reasons, num_returns,
                                p=[0.15,0.2,0.15,0.2,0.1,0.1,0.1]),
    'return_attempt_hour': np.random.randint(0, 24, num_returns),
    'pickup_available': np.random.choice([True,False], num_returns, p=[0.6,0.4])
})
returns.to_csv(os.path.join(output_path, 'returns.csv'), index=False)
print(f"returns:   {len(returns):,} rows saved")

# --- deliveries ---
deliveries = pd.DataFrame({
    'delivery_id': [f'DEL{i:06d}' for i in range(N*2)],
    'order_id': orders['order_id'].values,
    'pincode': np.random.choice(pincodes, N*2),
    'delivery_success': np.random.choice([True,False], N*2, p=[0.78,0.22]),
    'transit_hours': np.random.randint(1, 96, N*2),
    'quality_complaint': np.random.choice([True,False], N*2, p=[0.12,0.88]),
    'temperature_breach': np.random.choice([True,False], N*2, p=[0.08,0.92])
})
deliveries['is_rural'] = deliveries['pincode'].map(pincode_rural)
deliveries.to_csv(os.path.join(output_path, 'deliveries.csv'), index=False)
print(f"deliveries:{len(deliveries):,} rows saved")

# --- inventory snapshots ---
dates = pd.date_range('2024-01-01', periods=365, freq='D')
inventory = pd.DataFrame({
    'snapshot_date': dates.repeat(10),
    'product_id': list(np.random.choice(products['product_id'], 3650)),
    'stock_level': np.random.randint(0, 600, 3650),
    'stockout_event': np.random.choice([True,False], 3650, p=[0.08,0.92])
})
inventory.to_csv(os.path.join(output_path, 'inventory.csv'), index=False)
print(f"inventory: {len(inventory):,} rows saved")

print("\nAll 6 datasets created successfully!")
print(f"Saved to: {output_path}")

customers: 5,000 rows saved
products:  500 rows saved
orders:    10,000 rows saved
returns:   900 rows saved
deliveries:10,000 rows saved
inventory: 3,650 rows saved

All 6 datasets created successfully!
Saved to: C:\Users\chant\ecommerce-analytics\data\raw
